In [3]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
import warnings
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import scale
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error ,r2_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn import model_selection
from sklearn.linear_model import Ridge ,Lasso,ElasticNet
import statsmodels.api as sm
from warnings import filterwarnings
filterwarnings('ignore')
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV

print("Burada kütüphaneleri tanımladık")

Burada kütüphaneleri tanımladık


In [4]:
df=pd.read_csv("forestfires.csv")
df= df.iloc[:,1:len(df)]    
print("Burada veri okuma islemini gerceklestiriyoruz csv'den okunuyor ")

FileNotFoundError: [Errno 2] No such file or directory: 'forestfires.csv'

In [ ]:
df.head()


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Read the CSV file
df = pd.read_csv("./forestfires.csv")  # Ensure the file path is correct
df = df.dropna()

# Create dummy variables for 'month' and 'day'
dms = pd.get_dummies(df[['month', 'day']])

# Define target variable
y = df["area"]

# Drop 'area', 'month', and 'day' from X
X_ = df.drop(['area', 'month', 'day'], axis=1).astype('float')

# Concatenate dummy variables with X
X = pd.concat([X_, dms], axis=1)

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=99)


In [ ]:
df.info()
print("Veri setinin veri tipi ve öğe sayısı")

In [ ]:
df.describe()

In [ ]:
df.isnull()

In [ ]:
# Sadece sayısal sütunları seçtik
numerical_columns = df.select_dtypes(include=['float64', 'int64'])
korelasyon = numerical_columns.corr()

# Korelasyon matrisini görselleştirdik
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))
sns.heatmap(korelasyon, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Korelasyon Matrisi")
plt.show()


In [ ]:
print(df.describe())

In [ ]:
print(df.isnull().sum())
print("Veri setinin eksik değerleri")


In [ ]:
import matplotlib.pyplot as plt
sns.pairplot(df)
plt.show()

In [ ]:

# veri setimizin %75'ini eğitime ayırırken kalanları teste ayıroyuruz.
#random_state=123 parametresi, veri bölmesinin her çalıştırmada aynı sonucu vermesini sağlar, böylece kodun yeniden çalıştırılmasında tutarlılık sağlanır.
X_train, X_test, y_train, y_test =train_test_split(X, y, test_size=0.25,random_state=123)

print("Eğitim ve test setimiz belirlendi")

In [ ]:
print("Bizler KNN, RF ve ExtraTree yöntemlerini kullanıyoruz")
print("Oncelikle KNN için en iyi paremetreleri arıyoruz. CV=5 ve Grid Search ile parametre eniyilemesi")
###########################################
#burada KNN with Grid Search
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.neighbors import KNeighborsRegressor  
from sklearn.metrics import mean_squared_error, r2_score 
# Hiperparametre ızgarası
param_grid = {
    'n_neighbors': [3, 5, 10, 15, 20],      # Komşu sayısı
     # Mesafe metrikleri
    'p': [1,2]                           # Minkowski metriği için p değerleribiz mi verdik 1 2 yi
}

# KNN model
knn_model = KNeighborsRegressor()

# GridSearchCV
grid_search = GridSearchCV(
    estimator=knn_model,
    param_grid=param_grid,
    scoring='neg_mean_squared_error',  # Performans metriği
    cv=5,                             # 5 katlı çapraz doğrulama                     
)

grid_search.fit(X_train, y_train)
# oncelikle buraya kadar calıstırıyoruz ve elde ettigimiz parametelere yeniden calisacagiz
# En iyi parametreler
print("KNN için en iyi parametreler:", grid_search.best_params_)



In [5]:
## knn icin en iyi parametre olarak bulduklarımızı kullanacagız
print("simdi knn icin buldugumuz en iyi parametrelerle, eğitim ve test basarımızı MSE ve r2 uzerınden hesaplıyoruz")
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor

# KNN modelini en iyi parametrelerle oluşturuyoruz
knn = KNeighborsRegressor(n_neighbors=20, p=1)

# Eğitim verisini kullanarak modeli eğitiyoruz
knn.fit(X_train, y_train)

# Eğitim seti başarısı
y_train_pred = knn.predict(X_train)
train_mse = mean_squared_error(y_train, y_train_pred),
train_r2 = r2_score(y_train, y_train_pred)
print("Eğitim MSE:", train_mse)
print("Eğitim R²:", train_r2)

# Test seti başarısı
y_pred = knn.predict(X_test)
test_mse = mean_squared_error(y_test, y_pred)
test_r2 = r2_score(y_test, y_pred)
print("Test MSE:", test_mse)
print("Test R²:", test_r2)


simdi knn icin buldugumuz en iyi parametrelerle, eğitim ve test basarımızı MSE ve r2 uzerınden hesaplıyoruz


NameError: name 'X_train' is not defined

In [ ]:
print("RandomForest")


In [ ]:
print("Oncelikle Rasgele Ormanlar Regresyonu için en iyi paremetreleri arıyoruz. CV=5 ve Grid Search ile parametre eniyilemesi")
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestRegressor
param_grid = [
    {'bootstrap': [False,True], 'n_estimators': [3, 10], 'max_features': [1, 2, 3]},
  ]

forest_reg = RandomForestRegressor(random_state=123)
rfr_grid_search = GridSearchCV(forest_reg, param_grid, cv=5,
                           scoring='neg_mean_squared_error',
                           return_train_score=True)
rfr_grid_search.fit(X_train,y_train)

print("Random Forests için en iyi parametreler:", rfr_grid_search.best_params_)


In [ ]:
## simdi RFR icin en iyi paraemtrelerle testlerimizi yapıyoruz 
print("simdi RFR icin buldugumuz en iyi parametrelerle, eğitim ve test basarımızı MSE ve r2 uzerınden hesaplıyoruz")
from sklearn.metrics import mean_squared_error, r2_score


# rfr modelini en iyi parametrelerle oluşturuyoruz
rfr = RandomForestRegressor(bootstrap=True, max_features=1, n_estimators=10)

# Eğitim verisini kullanarak modeli eğitiyoruz
rfr.fit(X_train, y_train)

# Eğitim seti başarısı
y_train_pred = rfr.predict(X_train)
train_mse = mean_squared_error(y_train, y_train_pred),
train_r2 = r2_score(y_train, y_train_pred)
print("Eğitim MSE:", train_mse)
print("Eğitim R²:", train_r2)

# Test seti başarısı
y_pred = rfr.predict(X_test)
test_mse = mean_squared_error(y_test, y_pred)
test_r2 = r2_score(y_test, y_pred)
print("Test MSE:", test_mse)
print("Test R²:", test_r2)

In [ ]:
print("Oncelikle ExtraTree Regresyonu için en iyi paremetreleri arıyoruz. CV=5 ve Grid Search ile parametre eniyilemesi")
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import ExtraTreesRegressor
param_grid = [
    {'n_estimators': [10,100], 'max_features': [1, 2, 3]},
 ]

ext_reg = ExtraTreesRegressor(random_state=123)
ext_grid_search = GridSearchCV(ext_reg, param_grid, cv=5,
                           scoring='neg_mean_squared_error',
                           return_train_score=True)
ext_grid_search.fit(X_train,y_train)

print("Exttra Tree  için en iyi parametreler:", ext_grid_search.best_params_)

In [ ]:
    ## simdi Ext Tree icin en iyi paraemtrelerle testlerimizi yapıyoruz 

    print("simdi Extra icin buldugumuz en iyi parametrelerle, eğitim ve test basarımızı MSE ve r2 uzerınden hesaplıyoruz")
    from sklearn.metrics import mean_squared_error, r2_score
    # ext modelini en iyi parametrelerle o=luşturuyoruz0
    ext = ExtraTreesRegressor(max_features=2, n_estimators=100)

    # Eğitim verisini kullanarak modeli eğitiyoruz
    ext.fit(X_train, y_train)

    # Eğitim seti başarısı
    y_train_pred = ext.predict(X_train)
    train_mse = mean_squared_error(y_train, y_train_pred),
    train_r2 = r2_score(y_train, y_train_pred)
    print("Eğitim MSE:", train_mse)
    print("Eğitim R²:", train_r2)

    # Test seti başarısı
    y_pred = ext.predict(X_test)
    test_mse = mean_squared_error(y_test, y_pred)
    test_r2 = r2_score(y_test, y_pred)
    print("Test MSE:", test_mse)
    print("Test R²:", test_r2)